In [ ]:
import pandas as pd
import numpy as np



## Extracting features of 2018 movies from Wikipedia

In [ ]:
import pandas as pd
import requests

# Wikipedia link
link = "https://en.wikipedia.org/wiki/List_of_American_films_of_2018"

# Add User-Agent to avoid 403 error
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(link, headers=headers)

# Parse tables from the page
tables = pd.read_html(response.text, header=0)

# Select required tables
df1 = tables[2]
df2 = tables[3]
df3 = tables[4]
df4 = tables[5]

# Combine all into one DataFrame
df = pd.concat([df1, df2, df3, df4], ignore_index=True)

print(df.head())


In [ ]:
# Using pd.concat() instead of append()
import pandas as pd

# This creates a single DataFrame by concatenating all DataFrames vertically
df = pd.concat([df1, df2, df3, df4], ignore_index=True)

# The above is equivalent to the nested append() calls but is more efficient and works in current pandas versions

In [ ]:
df

In [ ]:
from tmdbv3api import TMDb
import json
import requests
tmdb = TMDb()
tmdb.api_key = '5fd684aa7ffec0db4567c813532b6f3e'

In [ ]:
from tmdbv3api import Movie
tmdb_movie = Movie()
import requests, time

def get_genre(title):
    try:
        result = tmdb_movie.search(title)
        movie_id = result[0].id
        url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={tmdb.api_key}"
        response = requests.get(url, timeout=10)   # timeout added
        data_json = response.json()
        if "genres" in data_json:
            return [g["name"] for g in data_json["genres"]]
    except Exception as e:
        print(f"Error fetching {title}: {e}")
        time.sleep(1)  # small delay before retry
        return None


In [ ]:
df['genres'] = df['Title'].map(lambda x: get_genre(str(x)))

In [ ]:
df

In [ ]:
df_2018 = df[['Title','Cast and crew','genres']]

In [ ]:
df_2018

In [ ]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x:
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

In [ ]:
df_2018['director_name'] = df_2018['Cast and crew'].map(lambda x: get_director(x))

In [ ]:
def get_actor1(x):
    return ((x.split("screenplay); ")[-1]).split(", ")[0])

In [ ]:
df_2018['actor_1_name'] = df_2018['Cast and crew'].map(lambda x: get_actor1(x))

In [ ]:
def get_actor2(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 2:
        return np.NaN
    else:
        return ((x.split("screenplay); ")[-1]).split(", ")[1])

In [ ]:
df_2018['actor_2_name'] = df_2018['Cast and crew'].map(lambda x: get_actor2(x))

In [ ]:
def get_actor3(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 3:
        return np.NaN
    else:
        return ((x.split("screenplay); ")[-1]).split(", ")[2])

In [ ]:
df_2018['actor_3_name'] = df_2018['Cast and crew'].map(lambda x: get_actor3(x))

In [ ]:
df_2018

In [ ]:
df_2018 = df_2018.rename(columns={'Title':'movie_title'})

In [ ]:
new_df18 = df_2018.loc[:,['director_name','actor_1_name','actor_2_name','actor_3_name','genres','movie_title']]

In [ ]:
new_df18

In [ ]:
new_df18['actor_2_name'] = new_df18['actor_2_name'].replace(np.nan, 'unknown')
new_df18['actor_3_name'] = new_df18['actor_3_name'].replace(np.nan, 'unknown')

In [ ]:
new_df18['movie_title'] = new_df18['movie_title'].str.lower()

In [ ]:
new_df18['comb'] = new_df18['actor_1_name'] + ' ' + new_df18['actor_2_name'] + ' '+ new_df18['actor_3_name'] + ' '+ new_df18['director_name'] +' ' + new_df18['genres']

In [ ]:
new_df18

## Extracting features of 2019 movies from Wikipedia

In [ ]:
link = "https://en.wikipedia.org/wiki/List_of_American_films_of_2019"
df1 = pd.read_html(link, header=0)[2]
df2 = pd.read_html(link, header=0)[3]
df3 = pd.read_html(link, header=0)[4]
df4 = pd.read_html(link, header=0)[5]

In [ ]:
df = df1.append(df2.append(df3.append(df4,ignore_index=True),ignore_index=True),ignore_index=True)

In [ ]:
df

In [ ]:
df['genres'] = df['Title'].map(lambda x: get_genre(str(x)))

In [ ]:
df_2019 = df[['Title','Cast and crew','genres']]

In [ ]:
df_2019

In [ ]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x:
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

In [ ]:
df_2019 = df[['Title','Cast and crew']].copy()


In [ ]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x:
        return x.split(" (directors)")[0]
    elif " (director/screenplay)" in x:
        return x.split(" (director/screenplay)")[0]
    else:
        return None

df_2019['director_name'] = df_2019['Cast and crew'].map(lambda x: get_director(str(x)))

# ---- actor extractors ----
def get_actor1(x):
    return ((x.split("screenplay); ")[-1]).split(", ")[0])

def get_actor2(x):
    parts = (x.split("screenplay); ")[-1]).split(", ")
    return parts[1] if len(parts) > 1 else np.NaN

def get_actor3(x):
    parts = (x.split("screenplay); ")[-1]).split(", ")
    return parts[2] if len(parts) > 2 else np.NaN

df_2019['actor_1_name'] = df_2019['Cast and crew'].map(lambda x: get_actor1(str(x)))
df_2019['actor_2_name'] = df_2019['Cast and crew'].map(lambda x: get_actor2(str(x)))
df_2019['actor_3_name'] = df_2019['Cast and crew'].map(lambda x: get_actor3(str(x)))

# ✅ rename title column
df_2019 = df_2019.rename(columns={'Title':'movie_title'})

In [ ]:
df_2019 = df_2019.rename(columns={'Title':'movie_title'})

In [ ]:
df_2019['genres'] = np.nan   # or fill with "Unknown"
new_df19 = df_2019.loc[:, ['director_name','actor_1_name','actor_2_name','actor_3_name','genres','movie_title']]



In [ ]:
new_df19['actor_2_name'] = new_df19['actor_2_name'].replace(np.nan, 'unknown')
new_df19['actor_3_name'] = new_df19['actor_3_name'].replace(np.nan, 'unknown')

In [ ]:
new_df19['movie_title'] = new_df19['movie_title'].str.lower()

In [ ]:
new_df19['comb'] = new_df19['actor_1_name'] + ' ' + new_df19['actor_2_name'] + ' '+ new_df19['actor_3_name'] + ' '+ new_df19['director_name'] +' ' + new_df19['genres']

In [ ]:
new_df19

In [ ]:
import pandas as pd

my_df = pd.concat([new_df18, new_df19], ignore_index=True)


In [ ]:
my_df

In [ ]:
old_df = pd.read_csv('../datasets/new_data.csv')

In [ ]:
old_df

In [ ]:
final_df = pd.concat([old_df, my_df], ignore_index=True)


In [ ]:
final_df

In [ ]:
final_df.isna().sum()

In [ ]:
final_df = final_df.dropna(how='any')

In [ ]:
final_df.isna().sum()

In [ ]:
final_df.to_csv('../datasets/final_data.csv',index=False)